In [2]:
import torch

data = torch.load("../activations/bbq.pt", weights_only=False)

# SAE activations are stored as sparse tensors — convert to dense
sae_activations = [act.to_dense() for act in data["sae_activations"]]
generations     = data["generations"]
categories      = data["categories"]
model_config    = data["model_config"]
sae_config      = data["sae_config"]

print(f"Loaded {len(sae_activations)} samples")
print(f"Model: {model_config['model_name']}")
print(f"SAE:   layer {sae_config['layer']}, width {sae_config['width']}, L0 {sae_config['l0']}")

Loaded 900 samples
Model: google/gemma-3-27b-it
SAE:   layer 31, width 65k, L0 medium


In [6]:
# Exp 1. Aggregation (consistency_max) -> Normalization (standard_scaler)

from tqdm import tqdm
from src.aggregator import Aggregator
from src.denoiser import Denoiser
from src.configs import SAEConfig
from src.neuronpedia_client import NeuronpediaClient, build_sae_id
from src.feature import Feature

aggregator = Aggregator()
aggregated = torch.stack([
    aggregator.max(act)
    for act in tqdm(sae_activations, desc="Aggregating")
])

print(f"Aggregated matrix shape: {aggregated.shape}")

# Build Neuronpedia client and denoiser
model_id = model_config["model_name"].split("/")[-1]
sae_cfg = SAEConfig(
    repo_id=sae_config["repo_id"],
    sae_type=sae_config["sae_type"],
    layer=sae_config["layer"],
    width=sae_config["width"],
    l0=sae_config["l0"],
)
client = NeuronpediaClient(model_id=model_id, sae_id=build_sae_id(sae_cfg))
denoiser = Denoiser(neuronpedia_client=client)

normalised = denoiser.standard_scaler(aggregated)

PROMPT_IDX = 231
TOP_K = 30

normalised = denoiser.global_idf(normalised[PROMPT_IDX].unsqueeze(0))

top_strengths, top_indices = normalised.squeeze(0).topk(TOP_K)
features = Feature.from_activations(top_indices, top_strengths, client)

print(f"Prompt #{PROMPT_IDX}  |  category: {categories[PROMPT_IDX]}")
print(f"\nGeneration:\n{generations[PROMPT_IDX]}\n")
print(f"Top {TOP_K} SAE features (standard-scaled):")
for f in features:
    desc = f.description or "(no description)"
    print(f"  Feature {f.feature_idx:>6d}  score={f.strength:+.3f}  ->  {desc}")

Aggregating: 100%|██████████| 900/900 [02:01<00:00,  7.43it/s]


Aggregated matrix shape: torch.Size([900, 65536])


global_idf: fetching frac_nonzero:   3%|▎         | 1371/43240 [06:15<3:11:12,  3.65it/s] 


KeyboardInterrupt: 